# NB09 — Musanga as an indicator of disturbance and secondary succession
V5 · CPU · UAV-derived binary fractional cover from NB07; no spectral predictors or NB08 dependency.
Main question: does present Musanga abundance reveal disturbance history, and how does this depend on spatial support, disturbance extent and repeated disturbance?
This is a cross-sectional chronosequence, not longitudinal observation. No mapped event is not evidence of no disturbance. Forests are separate case studies, not independent replicates of a common regional succession curve.
Run 00–07. Native caches are preserved; new ecological outputs are isolated in a versioned subfolder. No old data are deleted.


In [ ]:
#@title 00 — Mount Drive, verify UAV inputs and study region
%pip -q install earthengine-api rasterio geopandas pyarrow scipy pandas matplotlib
from pathlib import Path
import os,json,hashlib,math,time
from collections import defaultdict
from contextlib import ExitStack
import numpy as np,pandas as pd,geopandas as gpd,matplotlib.pyplot as plt
import rasterio,ee
from rasterio import Affine
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from scipy.stats import spearmanr
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final/V5'),Path('/content/drive/MyDrive/Musanga/Final/V5')] if (p.parent/'Data').is_dir() and p.is_dir()),None)
assert ROOT is not None
SITES=['Baego','MOS','Yangambi','Yoko'];YEARS=dict(Baego=2023,MOS=2024,Yangambi=2024,Yoko=2020)
PROJECT='musanga-511110';CRS='EPSG:32635';SUPPORT=.90;SUB=30;TILE=64
GID='UMD/hansen/global_forest_change_2025_v1_13';JID='projects/JRC/TMF/v1_2025/AnnualChanges'
FORCE_REBUILD=False
def digest(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(p,x):
 tmp=p.with_suffix('.partial');tmp.write_text(json.dumps(x,indent=2,default=str));os.replace(tmp,p)
def atomic_parquet(p,x):
 tmp=p.with_suffix('.partial');x.to_parquet(tmp,index=False);os.replace(tmp,p)
NB07=ROOT/'Outputs/07_RGB_unified_production_v5';M=json.loads((NB07/'07_production_mosaic_manifest.json').read_text())
assert M['complete'] and M['schema']=='musanga_v5_nb07_rgb_mosaics' and len(M['mosaics'])==6
MOS=defaultdict(list);meta=[]
for item in M['mosaics']:
 p=NB07/'reconstructed_mosaics'/Path(item['output']).name
 assert p.is_file(),f'Missing NB07 mosaic: {p}'
 rec=json.loads((p.parent/(item['name']+'_reconstruction.json')).read_text())
 assert rec['status']=='complete' and rec['digest']==item['digest'] and rec['model']==M['model_sha'] and rec['policy']==M['policy_sha']
 with rasterio.open(p) as ds:
  assert ds.count==2 and ds.nodata==-9999 and ds.crs==rasterio.crs.CRS.from_string(CRS)
  assert ds.tags().get('signature')==item['digest'] and ds.width==item['width'] and ds.height==item['height']
  assert ds.descriptions==('Musanga probability','Musanga binary at 0.5')
  b=list(ds.bounds)
 MOS[item['site']].append((item['name'],p,b));meta.append(dict(name=item['name'],site=item['site'],signature=item['digest'],size=p.stat().st_size,bounds=b))
assert set(MOS)==set(SITES)
for s in SITES:MOS[s].sort(key=lambda x:x[0])
raw=json.loads((ROOT/'Outputs/00_input_audit/00_input_manifest.json').read_text())
rp=Path(raw['roi'])
if not rp.is_file():rp=ROOT.parent/'Data/Ortho_RGB'/rp.name
assert rp.is_file()
roi=unary_union(gpd.read_file(rp).to_crs(CRS).geometry).buffer(-100.)
assert roi.is_valid and not roi.is_empty
GEOM={s:roi.intersection(unary_union([box(*b) for _,_,b in MOS[s]])) for s in SITES}
assert all(not g.is_empty for g in GEOM.values())
def init_ee():
 try:ee.Initialize(project=PROJECT)
 except Exception:ee.Authenticate();ee.Initialize(project=PROJECT)
print('Verified six NB07 mosaics; no new UAV inference.')


In [ ]:
#@title 01 — Obtain and verify the original Hansen and JRC lattices
init_ee()
G=ee.Image(GID);J=ee.ImageCollection(JID)
def projinfo(img,band):
 p=img.select(band).projection().getInfo()
 a=Affine(*p['transform']);assert abs(a.b)<1e-12 and abs(a.d)<1e-12 and a.a>0 and a.e<0,p
 return p['crs'],a
def samegrid(p,q):
 c,a=p;d,b=q
 if c!=d or not np.allclose([a.a,a.b,a.d,a.e],[b.a,b.b,b.d,b.e],rtol=1e-9,atol=1e-12):return False
 x,y=(~a)*(b.c,b.f)
 return abs(x-round(x))<1e-5 and abs(y-round(y))<1e-5
PG=projinfo(G,'lossyear')
for band in ['datamask','treecover2000']:assert samegrid(PG,projinfo(G,band))
GRIDS={};IMAGES={};REGIONS={};TILES={}
for s in SITES:
 y=YEARS[s];g=GEOM[s]
 lonlat=gpd.GeoSeries([g],crs=CRS).to_crs('EPSG:4326').iloc[0]
 region=ee.Geometry.Rectangle(list(lonlat.bounds),proj='EPSG:4326',geodesic=False)
 jc=J.filterBounds(region).sort('system:index');ids=jc.aggregate_array('system:index').getInfo()
 assert ids,f'No JRC source tiles at {s}'
 bands=[f'Dec{k}' for k in range(1990,y)]
 js=jc.toList(len(ids));pj=projinfo(ee.Image(js.get(0)),bands[0])
 # Verify every selected annual source band on every intersecting original tile.
 for i in range(len(ids)):
  src=ee.Image(js.get(i));info=src.select(bands).getInfo()['bands']
  for b in info:
   assert samegrid(pj,(b['crs'],Affine(*b['crs_transform']))),(s,b)
 # Use source projection rather than default one-degree mosaic projection.
 jimg=jc.select(bands).mosaic().setDefaultProjection(pj[0],list(pj[1])[:6]).unmask(255).toUint8()
 land=G.select('datamask');loss=G.select('lossyear').unmask(0)
 gimg=ee.Image.cat([land.unmask(255),loss.updateMask(land.eq(1)).unmask(255),G.select('treecover2000').unmask(255)]).toUint8()
 for tag,p,img in [('Hansen',PG,gimg),('JRC',pj,jimg)]:
  c,a=p;GRIDS[s,tag]=p;IMAGES[s,tag]=img
  geom=gpd.GeoSeries([g],crs=CRS).to_crs(c).iloc[0];REGIONS[s,tag]=geom
  x0,y0,x1,y1=geom.bounds
  cl,rt=(~a)*(x0,y1);cr,rb=(~a)*(x1,y0)
  tiles=[]
  for row in range(math.floor(rt/TILE)*TILE,math.ceil(rb/TILE)*TILE,TILE):
   for col in range(math.floor(cl/TILE)*TILE,math.ceil(cr/TILE)*TILE,TILE):
    l,t=a*(col,row);r,b=a*(col+TILE,row+TILE)
    if geom.intersects(box(l,b,r,t)):tiles.append((row,col))
  assert tiles;TILES[s,tag]=tiles
  print(s,tag,c,list(a)[:6],len(tiles),'tiles')
CFG=dict(version='native_ecology_v1',sources=[GID,JID],years=YEARS,roi_sha=hashlib.sha256(roi.wkb).hexdigest(),mosaics=meta,model=M['model_sha'],policy=M['policy_sha'],support=SUPPORT,sub=SUB,tile=TILE,grids={f'{s}_{t}':[c,list(a)[:6]] for (s,t),(c,a) in GRIDS.items()})
OUT=ROOT/'Outputs/09_ecology_v5'/digest(CFG)[:20];OUT.mkdir(parents=True,exist_ok=True)
CACHE=OUT/'tiles';CACHE.mkdir(exist_ok=True);atomic_json(OUT/'config.json',CFG)
print('Native-grid output:',OUT)


In [ ]:
#@title 02 — Aggregate UAV cover directly to native pixels and attach disturbance histories
def native_tile(s,tag,row,col,stack):
 c,a=GRIDS[s,tag];tr=a*Affine.translation(col,row)
 # Approximate area integration by equally spaced subpixel centres within each native footprint.
 hi=tr*Affine.scale(1/SUB,1/SUB);size=TILE*SUB
 inside=rasterize([(REGIONS[s,tag],1)],out_shape=(size,size),transform=hi,fill=0,dtype='uint8').astype(bool)
 def reduce(z):return z.reshape(TILE,SUB,TILE,SUB).sum((1,3))
 best=np.zeros((TILE,TILE),dtype=np.int32);positive=best.copy();source=np.full(best.shape,-1,dtype=np.int16)
 for i,(name,p,b) in enumerate(MOS[s]):
  ds=stack[name]
  with WarpedVRT(ds,crs=c,transform=hi,width=size,height=size,resampling=Resampling.nearest,src_nodata=-9999,nodata=-9999,warp_mem_limit=128) as vrt:
   z=vrt.read(2)
  valid=inside&((z==0)|(z==1));nv=reduce(valid);npix=reduce(valid&(z==1))
  better=nv>best;best[better]=nv[better];positive[better]=npix[better];source[better]=i
 keep=best>=math.ceil(SUPPORT*SUB*SUB)
 if not keep.any():return pd.DataFrame()
 # Explicit PixelGrid affine; Earth Engine returns structured NPY, not a resampled arbitrary 30m UTM tile.
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:
   arr=ee.data.computePixels(dict(expression=IMAGES[s,tag],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 assert arr.shape==(TILE,TILE)
 rr,cc=np.where(keep);xx,yy=tr*(cc+.5,rr+.5)
 d=pd.DataFrame(dict(site=s,product=tag,row=rr+row,col=cc+col,x_native=xx,y_native=yy,survey_year=YEARS[s],support=best[keep]/(SUB*SUB),cover_pp=100*positive[keep]/best[keep],mosaic=[MOS[s][i][0] for i in source[keep]]))
 y=YEARS[s]
 if tag=='Hansen':
  assert set(arr.dtype.names)=={'datamask','lossyear','treecover2000'},arr.dtype.names
  mask=arr['datamask'][keep];loss=arr['lossyear'][keep].astype(int);tc=arr['treecover2000'][keep]
  ok=(mask==1)&(loss>=0)&(loss<=25)
  prior=ok&(loss>0)&(2000+loss<y)
  d['history_valid']=ok;d['loss_year']=np.where(prior,2000+loss,np.nan)
  d['age_since_loss']=y-d.loss_year
  d['loss_status']=np.select([~ok,prior,ok&(loss>0)&(2000+loss>=y)],['invalid','prior loss','survey year or later loss'],default='no detected loss')
  d['treecover2000']=np.where(tc<=100,tc,np.nan)
 else:
  years=np.arange(1990,y);names=[f'Dec{k}' for k in years]
  assert set(arr.dtype.names)==set(names),arr.dtype.names
  hist=np.stack([arr[b][keep] for b in names],axis=0)
  ok=np.isin(hist,np.arange(1,7)).all(axis=0)
  # Only genuine transitions from forest to class 3, not each year spent deforested.
  entries=np.isin(hist[:-1],[1,2,4])&(hist[1:]==3)
  latest=np.max(np.where(entries,years[1:,None],0),axis=0)
  # Current uninterrupted class 4 episode, with directly preceding deforested state.
  run=np.cumprod((hist[::-1]==4).astype(np.int16),axis=0).sum(axis=0)
  onset=len(years)-run;prev=hist[np.maximum(onset-1,0),np.arange(len(d))]
  confirmed=ok&(run>0)&(onset>0)&(prev==3)
  d['history_valid']=ok;d['jrc_state']=np.where(ok,hist[-1],np.nan)
  d['deforestation_year']=np.where(ok&(latest>0),latest,np.nan)
  d['age_since_deforestation']=y-d.deforestation_year
  d['regrowth_onset_year']=np.where(confirmed,years[0]+onset,np.nan)
  d['age_since_regrowth']=y-d.regrowth_onset_year
  d['deforestation_status']=np.select([~ok,latest>0,(hist==3).any(axis=0)],['invalid','dated forest-to-deforested transition','deforested state without dated forest transition'],default='no detected deforestation')
  d['regrowth_status']=np.select([~ok,confirmed,run>0],['invalid','dated post-deforestation regrowth','other or left-censored regrowth'],default='not currently regrowth')
 return d
if (OUT/'completion.json').is_file() and json.loads((OUT/'completion.json').read_text()).get('complete') and all((OUT/f'{tag}_native_pixels.parquet').is_file() for tag in ['Hansen','JRC']) and not FORCE_REBUILD:
 print('LOAD completed native tables:',OUT)
else:
 all_tables={}
 for s in SITES:
  with ExitStack() as es:
   stack={name:es.enter_context(rasterio.open(p)) for name,p,b in MOS[s]}
   for tag in ['Hansen','JRC']:
    chunks=[]
    for i,(row,col) in enumerate(TILES[s,tag]):
     p=CACHE/f'{s}_{tag}_{row}_{col}.parquet'
     if p.is_file() and not FORCE_REBUILD:d=pd.read_parquet(p)
     else:
      d=native_tile(s,tag,row,col,stack);atomic_parquet(p,d)
     if len(d):chunks.append(d)
     print(s,tag,i+1,'/',len(TILES[s,tag]),'pixels',len(d),flush=True)
    assert chunks,(s,tag)
    all_tables[s,tag]=pd.concat(chunks,ignore_index=True)
 for tag in ['Hansen','JRC']:
  d=pd.concat([all_tables[s,tag] for s in SITES],ignore_index=True)
  assert not d.duplicated(['site','row','col']).any()
  assert d.cover_pp.between(0,100).all() and d.support.ge(SUPPORT).all()
  atomic_parquet(OUT/f'{tag}_native_pixels.parquet',d)
  display(d.groupby('site').agg(n=('cover_pp','size'),valid_history=('history_valid','sum'),mean_support=('support','mean')))
 

In [ ]:
#@title 03 — Four simple figures: raw pixel-level cover against age and JRC state
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
rows=[]
for d,col,label in [(H,'age_since_loss','Years since Hansen detected tree-cover loss'),(J9,'age_since_deforestation','Years since JRC forest-to-deforested transition'),(J9,'age_since_regrowth','Years since onset of JRC post-deforestation regrowth')]:
 fig,axs=plt.subplots(2,2,figsize=(11,8),sharex=True,sharey=True,layout='constrained')
 maxage=max(1,d[col].max()) if d[col].notna().any() else 1
 for s,ax in zip(SITES,axs.ravel()):
  g=d[d.site.eq(s)];q=g[g.history_valid&g[col].notna()]
  x=q[col];y=q.cover_pp
  rho=spearmanr(x,y).statistic if len(q)>2 and x.nunique()>1 and y.nunique()>1 else np.nan
  ax.scatter(x,y,s=5,alpha=.18,rasterized=True)
  ax.set(title=f'{s} | n={len(q):,}',xlabel=label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  rows.append(dict(site=s,variable=col,n=len(q),n_total=len(g),n_valid_history=int(g.history_valid.sum()),rho=rho))
 fig.savefig(OUT/f'{col}.png',dpi=220);plt.show();plt.close(fig)
names={1:'Undisturbed TMF',2:'Degraded TMF',3:'Deforested land',4:'Regrowth',5:'Water',6:'Other land cover'}
fig,axs=plt.subplots(2,2,figsize=(12,8),sharey=True,layout='constrained')
for s,ax in zip(SITES,axs.ravel()):
 g=J9[J9.site.eq(s)&J9.history_valid];groups=[];labels=[]
 for k,v in names.items():
  q=g[g.jrc_state.eq(k)].cover_pp
  if len(q):groups.append(q.to_numpy());labels.append(f'{v}\n(n={len(q):,})')
 if groups:ax.boxplot(groups,tick_labels=labels,showfliers=True)
 ax.tick_params(axis='x',labelrotation=30,labelsize=8);ax.set(title=s,ylabel='UAV Musanga cover (%)',ylim=(0,100))
fig.savefig(OUT/'JRC_state_cover.png',dpi=220);plt.show();plt.close(fig)
summary=pd.DataFrame(rows);summary.to_csv(OUT/'age_scatter_summary.csv',index=False);display(summary)
for d,col in [(H,'loss_status'),(J9,'deforestation_status'),(J9,'regrowth_status'),(J9,'jrc_state')]:
 q=d.groupby(['site',col],dropna=False).cover_pp.agg(n='size',mean_cover='mean',median_cover='median').reset_index()
 q.to_csv(OUT/f'{col}_summary.csv',index=False);display(q.round(2))
atomic_json(OUT/'completion.json',dict(complete=True,config=CFG,n_Hansen=len(H),n_JRC=len(J9),outputs=sorted(p.name for p in OUT.iterdir())))
print('NB09 NATIVE GRID COMPLETE',OUT)


## 04 — Full JRC history and native baseline
Annual JRC class transitions are proxies, not counts of independently validated events. Exclude survey-year events because their timing relative to UAV acquisition is unknown. Reuse the original native and degradation caches where compatible.


In [ ]:
#@title 04 — Native-grid annual cover statistics and latest JRC degradation age
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet');J9=pd.read_parquet(OUT/'JRC_native_pixels.parquet')
assert (OUT/'completion.json').is_file() and json.loads((OUT/'completion.json').read_text())['complete']
O4=OUT/'04_annual_ages_and_degradation_v1';O4.mkdir(exist_ok=True)
DGC=O4/'jrc_tiles';DGC.mkdir(exist_ok=True)
def degradation_tile(s,row,col,g):
 c,a=GRIDS[s,'JRC'];tr=a*Affine.translation(col,row)
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:arr=ee.data.computePixels(dict(expression=IMAGES[s,'JRC'],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 years=np.arange(1990,YEARS[s]);bands=[f'Dec{k}' for k in years]
 assert arr.shape==(TILE,TILE) and set(arr.dtype.names)==set(bands)
 rr=g.row.to_numpy(int)-row;cc=g.col.to_numpy(int)-col
 hist=np.stack([arr[b][rr,cc] for b in bands])
 ok=np.isin(hist,np.arange(1,7)).all(axis=0)
 assert np.array_equal(ok,g.history_valid.to_numpy())
 assert np.array_equal(np.where(ok,hist[-1],np.nan),g.jrc_state.to_numpy(),equal_nan=True)
 defo=np.max(np.where(np.isin(hist[:-1],[1,2,4])&(hist[1:]==3),years[1:,None],0),axis=0)
 assert np.array_equal(np.where(ok&(defo>0),defo,np.nan),g.deforestation_year.to_numpy(),equal_nan=True)
 entry=np.isin(hist[:-1],[1,4])&(hist[1:]==2)
 latest=np.max(np.where(entry,years[1:,None],0),axis=0)
 d=g[['site','row','col']].copy()
 d['degradation_year']=np.where(ok&(latest>0),latest,np.nan)
 d['age_since_degradation']=YEARS[s]-d.degradation_year
 d['n_degradation_entries']=np.where(ok,entry.sum(axis=0),np.nan)
 d['degradation_status']=np.select([~ok,latest>0,(hist==2).any(axis=0)],['invalid','dated forest-to-degraded transition','degraded state without dated forest transition'],default='no detected degradation')
 return d
if (O4/'completion.json').is_file() and json.loads((O4/'completion.json').read_text()).get('complete') and (O4/'JRC_native_degradation.parquet').is_file() and not FORCE_REBUILD:
 DJ=pd.read_parquet(O4/'JRC_native_degradation.parquet');assert len(DJ)==len(J9)
else:
 parts=[]
 for (s,row,col),g in J9.assign(tile_row=(J9.row//TILE)*TILE,tile_col=(J9.col//TILE)*TILE).groupby(['site','tile_row','tile_col']):
  row=int(row);col=int(col);p=DGC/f'{s}_{row}_{col}.parquet'
  if p.is_file() and not FORCE_REBUILD:q=pd.read_parquet(p)
  else:q=degradation_tile(s,row,col,g);atomic_parquet(p,q)
  assert q[['site','row','col']].reset_index(drop=True).equals(g[['site','row','col']].reset_index(drop=True))
  parts.append(q);print(s,row,col,'JRC pixels',len(q),flush=True)
 DJ=J9.merge(pd.concat(parts,ignore_index=True),on=['site','row','col'],validate='one_to_one')
 assert len(DJ)==len(J9)
 atomic_parquet(O4/'JRC_native_degradation.parquet',DJ)
def add_blocks(d,tag):
 q=d.copy();q['block500']=''
 for s,g in q.groupby('site'):
  xy=gpd.GeoSeries(gpd.points_from_xy(g.x_native,g.y_native),index=g.index,crs=GRIDS[s,tag][0]).to_crs(CRS)
  bx=np.floor(xy.x.to_numpy()/500).astype(int);by=np.floor(xy.y.to_numpy()/500).astype(int)
  q.loc[g.index,'block500']=[f'{x}_{y}' for x,y in zip(bx,by)]
 return q
HH=add_blocks(H,'Hansen');JJ=add_blocks(DJ,'JRC')
specs=[(HH,'age_since_loss','Hansen detected loss'),(JJ,'age_since_deforestation','JRC deforestation'),(JJ,'age_since_regrowth','JRC regrowth onset'),(JJ,'age_since_degradation','JRC degradation onset')]
tables=[]
for d,col,label in specs:
 q=d[d.history_valid&d[col].notna()].copy()
 assert (q[col]>0).all() and np.allclose(q[col],q[col].astype(int))
 q['age']=q[col].astype(int)
 a=q.groupby(['site','age']).agg(n=('cover_pp','size'),blocks=('block500','nunique'),mean=('cover_pp','mean'),median=('cover_pp','median'),q25=('cover_pp',lambda z:z.quantile(.25)),q75=('cover_pp',lambda z:z.quantile(.75))).reset_index()
 a['variable']=col;a['sufficient_support']=(a.n>=30)&(a.blocks>=5);tables.append(a)
 maxage=max(1,q.age.max()) if len(q) else 1
 fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  g=q[q.site.eq(s)];z=a[a.site.eq(s)]
  ax.scatter(g.age,g.cover_pp,s=3,alpha=.07,color='grey',rasterized=True)
  for robust,zz in z.groupby('sufficient_support'):
   ax.errorbar(zz.age,zz['median'],yerr=[zz['median']-zz.q25,zz.q75-zz['median']],fmt='o' if robust else 'o',mfc='C0' if robust else 'none',mec='C0',ecolor='C0',alpha=1 if robust else .45,label='Median / IQR' if robust else 'Sparse age: median / IQR')
  if len(z):ax.scatter(z.age,z['mean'],marker='x',s=25,color='C1',label='Mean')
  ax.set(title=f'{s} | n={len(g):,}',xlabel='Years since '+label,ylabel='UAV Musanga cover (%)',xlim=(0,maxage+1),ylim=(0,100))
  if len(z):ax.legend(fontsize=7)
 fig.savefig(O4/f'{col}_annual.png',dpi=200);plt.show();plt.close(fig)
summary=pd.concat(tables,ignore_index=True);summary.to_csv(O4/'annual_age_summary.csv',index=False);display(summary.round(2))
q=DJ.groupby(['site','degradation_status'],dropna=False).cover_pp.agg(n='size',mean='mean',median='median').reset_index()
q.to_csv(O4/'degradation_status_summary.csv',index=False);display(q.round(2))
# Strictly descriptive sensitivity: degradation-only history (no mapped deforested state ever)
# is not inferred from absent dated transition; current analysis reports all dated degradation events.
atomic_json(O4/'completion.json',dict(complete=True,n=len(DJ),source=str(OUT),definition='latest observed 1/4 to 2 transition; annual IQR not CI'))
print('04 COMPLETE',O4)


## 05 — Disturbance history and agreement
First and latest observed forest-to-disturbance transitions, recurrence, censoring and Hansen/JRC comparison. No spectral analysis. Cached history tiles are resumable. Product agreement is not validation.


In [ ]:
#@title 05 — Reconstruct JRC histories and establish a shared metric analysis support
from scipy.spatial import cKDTree
from scipy import ndimage
from collections import deque
from pyproj import Geod
ECO=OUT/'05_multiscale_patch_ecology_v1';ECO.mkdir(exist_ok=True)
HIST=ECO/'history_tiles';HIST.mkdir(exist_ok=True)
def metric(d,tag):
 q=d.copy()
 for s,g in q.groupby('site'):
  xy=gpd.GeoSeries(gpd.points_from_xy(g.x_native,g.y_native),index=g.index,crs=GRIDS[s,tag][0]).to_crs(CRS)
  q.loc[g.index,'x']=xy.x.to_numpy();q.loc[g.index,'y']=xy.y.to_numpy()
  c,a=GRIDS[s,tag]
  if rasterio.crs.CRS.from_string(c).is_geographic:
   geod=Geod(ellps='WGS84')
   # Native cells have constant angular dimensions; compute area once per native row.
   areas={}
   for row,gg in g.groupby('row'):
    lon=float(gg.x_native.iloc[0]);lat=float(gg.y_native.iloc[0]);dx=abs(a.a)/2;dy=abs(a.e)/2
    areas[row]=abs(geod.polygon_area_perimeter([lon-dx,lon+dx,lon+dx,lon-dx],[lat-dy,lat-dy,lat+dy,lat+dy])[0])
   q.loc[g.index,'area_m2']=g.row.map(areas).to_numpy()
  else:q.loc[g.index,'area_m2']=abs(a.a*a.e)
 q['w']=q.area_m2*q.support
 return q
def full_history_tile(s,row,col,g):
 c,a=GRIDS[s,'JRC'];tr=a*Affine.translation(col,row)
 grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,shearY=tr.d,scaleY=tr.e,translateY=tr.f),crsCode=c)
 for attempt in range(5):
  try:arr=ee.data.computePixels(dict(expression=IMAGES[s,'JRC'],fileFormat='NUMPY_NDARRAY',grid=grid));break
  except Exception:
   if attempt==4:raise
   time.sleep(2**attempt)
 years=np.arange(1990,YEARS[s]);rr=g.row.to_numpy(int)-row;cc=g.col.to_numpy(int)-col
 h=np.stack([arr[f'Dec{k}'][rr,cc] for k in years]);ok=np.isin(h,np.arange(1,7)).all(axis=0)
 assert np.array_equal(ok,g.history_valid.to_numpy())
 assert np.array_equal(np.where(ok,h[-1],np.nan),g.jrc_state.to_numpy(),equal_nan=True)
 deg=np.isin(h[:-1],[1,4])&(h[1:]==2);defo=np.isin(h[:-1],[1,2,4])&(h[1:]==3)
 event=deg|defo
 first=np.min(np.where(event,years[1:,None],9999),axis=0);last=np.max(np.where(event,years[1:,None],0),axis=0)
 q=g[['site','row','col']].copy()
 for name,val in [('n_entries',event.sum(axis=0)),('n_defo',defo.sum(axis=0)),('n_deg',deg.sum(axis=0)),('ever_defo',(h==3).any(axis=0)),('ever_deg',(h==2).any(axis=0)),('left_censored',np.isin(h[0],[2,3,4]))]:
  q[name]=np.where(ok,val,np.nan)
 q['first_year']=np.where(ok&(first!=9999),first,np.nan);q['last_year']=np.where(ok&(last>0),last,np.nan)
 return q
P=ECO/'JRC_full_history.parquet'
if P.is_file() and not FORCE_REBUILD:
 F=pd.read_parquet(P)
else:
 parts=[]
 for (s,row,col),g in J9.assign(tile_row=(J9.row//TILE)*TILE,tile_col=(J9.col//TILE)*TILE).groupby(['site','tile_row','tile_col']):
  row=int(row);col=int(col);p=HIST/f'{s}_{row}_{col}.parquet'
  if p.is_file() and not FORCE_REBUILD:q=pd.read_parquet(p)
  else:q=full_history_tile(s,row,col,g);atomic_parquet(p,q)
  assert q[['site','row','col']].reset_index(drop=True).equals(g[['site','row','col']].reset_index(drop=True))
  parts.append(q)
 F=pd.concat(parts,ignore_index=True);atomic_parquet(P,F)
assert len(F)==len(J9)
J=metric(DJ.merge(F,on=['site','row','col'],validate='one_to_one'),'JRC')
G=metric(H,'Hansen')
J['age_first']=J.survey_year-J.first_year;J['age_last']=J.survey_year-J.last_year
J['history_group']=np.select([~J.history_valid,J.left_censored.eq(1),J.n_entries.ge(2),J.n_entries.eq(1)],['invalid','left-censored','multiple entries','single entry'],default='no observed entry')
for d in [G,J]:
 d['block500']=(d.x//500).astype(int).astype(str)+'_'+(d.y//500).astype(int).astype(str)
display(J.groupby(['site','history_group']).cover_pp.agg(['size','mean','median']))
# Product agreement: nearest native centres, one-to-one mutual nearest within 25 m.
# This is a sensitivity comparison, not exact overlap or independent validation.
pairs=[]
for s in SITES:
 a=G[G.site.eq(s)&G.history_valid].reset_index(drop=True);b=J[J.site.eq(s)&J.history_valid].reset_index(drop=True)
 if not len(a) or not len(b):continue
 dist,i=cKDTree(b[['x','y']]).query(a[['x','y']]);_,rev=cKDTree(a[['x','y']]).query(b[['x','y']])
 k=np.where((dist<=25)&(rev[i]==np.arange(len(a))))[0]
 q=a.iloc[k][['site','row','col','x','y','cover_pp','loss_year','loss_status']].reset_index(drop=True)
 z=b.iloc[i[k]].reset_index(drop=True)
 for c in ['first_year','last_year','deforestation_year','n_entries','history_group']:q[c]=z[c]
 q['distance_m']=dist[k];pairs.append(q)
A=pd.concat(pairs,ignore_index=True) if pairs else pd.DataFrame()
if len(A):
 A['agreement']=np.select([A.loss_year.notna()&A.last_year.notna()&((A.loss_year-A.last_year).abs()<=1),A.loss_year.notna()&A.last_year.notna(),A.loss_year.notna(),A.last_year.notna()],['both within 1 yr','both different years','Hansen only','JRC only'],default='neither dated')
 atomic_parquet(ECO/'product_agreement.parquet',A)
 display(A.groupby(['site','agreement']).cover_pp.agg(['size','mean','median']))
print('Full history and agreement ready:',ECO)


## 06 — Scale and event fragments
Compare forest-specific chronosequences across fixed metric grids and native connected disturbance fragments. Retain no-event and zero-cover observations for context. Coherent-age curves require adequate observed support, dated-event fraction and narrow age spread. Equal-unit means and spatial-block bootstrap intervals are descriptive, not causal or independent regional inference. Native fragments can be clipped by UAV coverage; exact-year connectivity is conservative. Candidate list is for manual historical imagery review; no validation is claimed.


In [ ]:
#@title 06 — Multiscale and disturbance-patch ecology (offline)
SCALES=[30,60,90,150,300,500];AGE_BINS=[0,3,6,9,12,15,20,26,40]
MIN_GRID_SUPPORT=.5;MIN_EVENT_SHARE=.5;MAX_AGE_IQR=2
BOOT=400;SEED=2271
def wmean(v,w):
 v=np.asarray(v,float);w=np.asarray(w,float);k=np.isfinite(v)&np.isfinite(w)&(w>0)
 return np.average(v[k],weights=w[k]) if k.any() else np.nan
def wquant(v,w,p):
 v=np.asarray(v,float);w=np.asarray(w,float);k=np.isfinite(v)&np.isfinite(w)&(w>0)
 if not k.any():return np.nan
 v=v[k];w=w[k];i=np.argsort(v);return np.interp(p,np.cumsum(w[i])/w.sum(),v[i])
SPEC=[('Hansen_loss',G,'age_since_loss'),('JRC_defo',J,'age_since_deforestation'),('JRC_degradation',J,'age_since_degradation'),('JRC_regrowth',J,'age_since_regrowth'),('JRC_last',J,'age_last')]
def summarize(g,age):
 v=g[g.history_valid];e=v[v[age].notna()];w=g.w
 r=dict(n_pixels=len(g),area_m2=w.sum(),cover_pp=wmean(g.cover_pp,w),valid_fraction=v.w.sum()/w.sum(),event_fraction=e.w.sum()/v.w.sum() if len(v) else np.nan,age=wquant(e[age],e.w,.5),age_iqr=wquant(e[age],e.w,.75)-wquant(e[age],e.w,.25),x=wmean(g.x,w),y=wmean(g.y,w))
 for t in [5,10,20]:
  r[f'event_fraction_{t}yr']=v.loc[v[age].le(t),'w'].sum()/v.w.sum() if len(v) else np.nan
 if 'n_entries' in v:r['multiple_entry_fraction']=wmean(v.n_entries.ge(2),v.w);r['mean_entries']=wmean(v.n_entries,v.w)
 return r
if (ECO/'multiscale.parquet').is_file() and (ECO/'patches.parquet').is_file() and not FORCE_REBUILD:
 T=pd.read_parquet(ECO/'multiscale.parquet');P=pd.read_parquet(ECO/'patches.parquet')
else:
 rows=[]
 for label,d,age in SPEC:
  for scale in SCALES:
   # Centre assignment to fixed metric grids: approximate spatial support, no interpolation.
   # Native 30 m retained separately; 30 m binning can split native support at edges.
   q=d.assign(gx=(d.x//scale).astype(int),gy=(d.y//scale).astype(int))
   for (s,gx,gy),g in q.groupby(['site','gx','gy'],sort=False):
    r=summarize(g,age)
    r.update(site=s,variable=label,scale=scale,gx=gx,gy=gy,grid_support=min(1,r['area_m2']/scale**2))
    rows.append(r)
 T=pd.DataFrame(rows);T['eligible']=(T.grid_support>=MIN_GRID_SUPPORT)&(T.valid_fraction>=.9)
 T['coherent_age']=T.eligible&(T.event_fraction>=MIN_EVENT_SHARE)&(T.age_iqr<=MAX_AGE_IQR)
 T['block500']=(T.x//500).astype(int).astype(str)+'_'+(T.y//500).astype(int).astype(str)
 atomic_parquet(ECO/'multiscale.parquet',T)
 # Native same-year connected components (8-neighbour), restricted to valid observed UAV footprint.
 # A patch may be truncated by ROI; counts are observed fragments, not full landscape events.
 # Exact-year connectivity avoids transitive chaining of +/-1-year neighbours.
 patches=[]
 for label,d,age in SPEC:
  for s,g in d[d.history_valid&d[age].notna()].groupby('site'):
   g=g.reset_index(drop=True); lookup={(int(r),int(c)):i for i,(r,c) in enumerate(zip(g.row,g.col))}
   seen=set();pid=0;aa=g[age].to_numpy()
   for i in range(len(g)):
    if i in seen:continue
    seen.add(i);queue=deque([i]);idx=[]
    while queue:
     j=queue.popleft();idx.append(j);r=int(g.row.iloc[j]);c=int(g.col.iloc[j])
     for dr in [-1,0,1]:
      for dc in [-1,0,1]:
       k=lookup.get((r+dr,c+dc))
       if k is not None and k not in seen and aa[k]==aa[i]:seen.add(k);queue.append(k)
    z=g.iloc[idx];r=summarize(z,age);pid+=1
    patches.append(dict(**r,site=s,variable=label,patch_id=f'{s}_{label}_{pid}',patch_ha=r['area_m2']/10000,block500=f"{int(r['x']//500)}_{int(r['y']//500)}"))
 P=pd.DataFrame(patches)
 atomic_parquet(ECO/'patches.parquet',P)
def curve(d,groups):
 out=[];q=d.copy();q['age_bin']=pd.cut(q.age,AGE_BINS,right=False)
 rng=np.random.default_rng(SEED)
 for key,g in q.groupby(groups+['age_bin'],observed=True):
  # Equal unit weighting; blocks resampled, not individual pixels.
  b=g.groupby('block500').cover_pp.agg(['sum','count']).to_numpy(float)
  lo=hi=np.nan
  if len(b)>=5:
   z=b[rng.integers(0,len(b),(BOOT,len(b)))].sum(axis=1);lo,hi=np.quantile(z[:,0]/z[:,1],[.025,.975])
  out.append(dict(zip(groups+['age_bin'],key),age=g.age.median(),n=len(g),blocks=len(b),mean=g.cover_pp.mean(),median=g.cover_pp.median(),lo=lo,hi=hi))
 return pd.DataFrame(out)
C=curve(T[T.coherent_age],['site','variable','scale'])
pcs=[]
for m in [.5,1,2]:
 z=curve(P[P.patch_ha>=m],['site','variable']);z['min_patch_ha']=m;pcs.append(z)
PC=pd.concat(pcs,ignore_index=True)
C.to_csv(ECO/'multiscale_curves.csv',index=False);PC.to_csv(ECO/'patch_curves.csv',index=False)
def plot_curves(tab,key,values,name):
 if tab.empty:return
 for var in tab.variable.unique():
  fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,layout='constrained')
  for s,ax in zip(SITES,axs.ravel()):
   for val in values:
    z=tab[(tab.site==s)&(tab.variable==var)&(tab[key]==val)].sort_values('age')
    if z.empty:continue
    good=z.blocks>=5
    ax.plot(z.age,z['mean'],alpha=.65,label=str(val))
    ax.scatter(z.loc[good,'age'],z.loc[good,'mean'],s=22)
    ax.scatter(z.loc[~good,'age'],z.loc[~good,'mean'],facecolors='none',edgecolors='grey',s=22)
    ax.fill_between(z.age,z.lo,z.hi,alpha=.08)
   ax.set(title=s,xlabel='Years since mapped event',ylabel='Mean UAV Musanga cover (%)',ylim=(0,100))
   if ax.lines:ax.legend(title=key,fontsize=8)
  fig.savefig(ECO/f'{name}_{var}.png',dpi=180);plt.show();plt.close(fig)
plot_curves(C,'scale',SCALES,'scale')
plot_curves(PC,'min_patch_ha',[.5,1,2],'patch')
display(P.groupby(['site','variable']).patch_ha.agg(['size','median','max']))
# Landscape context: use every eligible grid including zero cover and no detected events.
# Show extent/frequency instead of forcing a single age onto mixed disturbance cohorts.
Q=T[T.eligible&T.scale.isin([90,150,300,500])].copy()
Q['extent_bin']=pd.cut(Q.event_fraction,[-.001,.01,.25,.5,.75,1.001],include_lowest=True)
L=Q.groupby(['site','variable','scale','extent_bin'],observed=True).cover_pp.agg(n='size',mean='mean',median='median').reset_index()
L.to_csv(ECO/'extent_context.csv',index=False)
for var in ['Hansen_loss','JRC_last']:
 fig,axs=plt.subplots(2,2,figsize=(12,8),sharex=True,sharey=True,layout='constrained')
 for s,ax in zip(SITES,axs.ravel()):
  for sc in [90,150,300,500]:
   z=L[(L.site==s)&(L.variable==var)&(L.scale==sc)]
   if len(z):ax.plot([v.mid for v in z.extent_bin],z['mean'],'o-',label=str(sc))
  ax.set(title=s,xlabel='Fraction with dated mapped event',ylabel='Mean UAV Musanga cover (%)',ylim=(0,100))
  if ax.lines:ax.legend(title='m')
 fig.savefig(ECO/f'extent_{var}.png',dpi=180);plt.show();plt.close(fig)
R=Q[Q.variable.eq('JRC_last')].copy();R['recurrence_bin']=pd.cut(R.multiple_entry_fraction,[-.001,.01,.25,.5,.75,1.001])
R.groupby(['site','scale','recurrence_bin'],observed=True).cover_pp.agg(['size','mean','median']).to_csv(ECO/'recurrence_context.csv')
# Independent imagery validation sample: at most one grid per 500 m block, stratified by
# age/absence and observed cover. Coordinates for visual assessment, not validated labels.
V=Q[(Q.variable=='JRC_last')&(Q.scale==150)].copy()
V['age_group']=np.select([V.event_fraction.eq(0),V.age.le(5),V.age.le(10)],['no dated entry','1–5','6–10'],default='older/mixed')
V['cover_group']=pd.cut(V.cover_pp,[-.01,20,50,100],labels=['low','medium','high'])
V=V.sample(frac=1,random_state=SEED);chosen=[]
for s,g in V.groupby('site'):
 used=set();count=0
 buckets=[list(z.index) for _,z in g.groupby(['age_group','cover_group'],observed=True)]
 while count<30 and any(buckets):
  for bucket in buckets:
   while bucket:
    i=bucket.pop();b=V.loc[i,'block500']
    if b in used:continue
    chosen.append(i);used.add(b);count+=1;break
   if count>=30:break
W=V.loc[chosen].copy()
if len(W):
 ll=gpd.GeoSeries(gpd.points_from_xy(W.x,W.y),index=W.index,crs=CRS).to_crs(4326)
 W['longitude']=ll.x;W['latitude']=ll.y
 for c in ['interpreted_event_year','interpreted_repeated_events','confidence','notes']:W[c]=''
W.to_csv(ECO/'historical_imagery_validation_candidates.csv',index=False)
atomic_json(ECO/'completion.json',dict(complete=True,scales=SCALES,patch_min_ha=[.5,1,2],grid_support=MIN_GRID_SUPPORT,event_share=MIN_EVENT_SHARE,age_iqr=MAX_AGE_IQR,n_grid=len(T),n_patches=len(P),validation_candidates=len(W),validation_completed=False))
print('NB09 ECOLOGY COMPLETE',ECO)


## 07 — Final ecological synthesis
Run after 06; alternatively run 00–01, then 07A–07C to load completed caches directly. No spectral predictors or new model fitting.
- Verify acquisition dates against authoritative flight logs; a file modification date is not a flight date. The current dates are provisional until verified. Changing them requires rebuilding 01–06.
- Test annual JRC reference-year sensitivity (survey −2 to +2, through 2025); later years are diagnostic only.
- Compare first/latest JRC events to Hansen, and construct agreement-filtered chronosequences and bounded temporal-span disturbance fragments.
- Primary landscape support 150 m, sensitivity 90/300/500 m. Spatial block bootstrap is descriptive; no regional inference from four sites.
- Produce one short Word report and six multi-panel figures, with numerical results and explicit limitations. No claims of validation from agreement alone.


In [ ]:
#@title 07A — Direct-load completed ecology and diagnose temporal alignment
from scipy.spatial import cKDTree
from collections import deque
from pyproj import Geod
ECO=OUT/'05_multiscale_patch_ecology_v1'
assert json.loads((ECO/'completion.json').read_text())['complete'],'Run 02–06 first'
FINAL=ECO/'07_final_synthesis_v1';FINAL.mkdir(exist_ok=True)
H=pd.read_parquet(OUT/'Hansen_native_pixels.parquet')
DJ=pd.read_parquet(OUT/'04_annual_ages_and_degradation_v1/JRC_native_degradation.parquet')
F=pd.read_parquet(ECO/'JRC_full_history.parquet')
T=pd.read_parquet(ECO/'multiscale.parquet')
A=pd.read_parquet(ECO/'product_agreement.parquet')
P=pd.read_parquet(ECO/'patches.parquet')
def metric7(d,tag):
 q=d.copy()
 for s,g in q.groupby('site'):
  c,a=GRIDS[s,tag]
  xy=gpd.GeoSeries(gpd.points_from_xy(g.x_native,g.y_native),index=g.index,crs=c).to_crs(CRS)
  q.loc[g.index,'x']=xy.x.to_numpy();q.loc[g.index,'y']=xy.y.to_numpy()
  if rasterio.crs.CRS.from_string(c).is_geographic:
   geod=Geod(ellps='WGS84');areas={}
   for row,gg in g.groupby('row'):
    lon=float(gg.x_native.iloc[0]);lat=float(gg.y_native.iloc[0]);dx=abs(a.a)/2;dy=abs(a.e)/2
    areas[row]=abs(geod.polygon_area_perimeter([lon-dx,lon+dx,lon+dx,lon-dx],[lat-dy,lat-dy,lat+dy,lat+dy])[0])
   q.loc[g.index,'area_m2']=g.row.map(areas).to_numpy()
  else:q.loc[g.index,'area_m2']=abs(a.a*a.e)
 q['w']=q.area_m2*q.support
 return q
G=metric7(H,'Hansen');J=metric7(DJ.merge(F,on=['site','row','col'],validate='one_to_one'),'JRC')
J['age_last']=J.survey_year-J.last_year
J['history_group']=np.select([~J.history_valid,J.left_censored.eq(1),J.n_entries.ge(2),J.n_entries.eq(1)],['invalid','left-censored','multiple entries','single entry'],default='no observed entry')
A=A.merge(G[['site','row','col','w','survey_year']],on=['site','row','col'],validate='one_to_one')
A['age']=A.survey_year-A.loss_year
A['delta_first']=A.loss_year-A.first_year;A['delta_last']=A.loss_year-A.last_year
for d in [G,J,A]:
 d['block500']=(d.x//500).astype(int).astype(str)+'_'+(d.y//500).astype(int).astype(str)
# Acquisition date verification: provide authoritative flight log dates here, not inferred TIFF timestamps.
# If dates differ from YEARS, change block 00 and rebuild; never silently mix cached chronosequences.
VERIFIED_FLIGHT_YEARS={} # e.g. {'Baego':2023,...} ONLY after checking original acquisition records
assert set(VERIFIED_FLIGHT_YEARS)<=set(SITES)
for s,y in VERIFIED_FLIGHT_YEARS.items():
 assert y==YEARS[s],f'{s}: correct YEARS in 00 and rebuild 01–06 before using these results'
dates=pd.DataFrame([dict(site=s,configured_UAV_year=YEARS[s],JRC_baseline_year=YEARS[s]-1,verified=s in VERIFIED_FLIGHT_YEARS) for s in SITES])
dates.to_csv(FINAL/'acquisition_alignment.csv',index=False);display(dates)
# JRC reference year sensitivity, preserving the exact same UAV footprint and native rows.
# A later JRC class is not an explanatory pre-UAV variable. Compare paired valid observations.
YDIR=FINAL/'jrc_reference_tiles';YDIR.mkdir(exist_ok=True)
YFILE=FINAL/'jrc_reference_years.parquet'
if YFILE.is_file() and not FORCE_REBUILD:
 Y=pd.read_parquet(YFILE)
else:
 parts=[]
 for s in SITES:
  g=J[J.site.eq(s)].copy();years=list(range(YEARS[s]-2,min(2025,YEARS[s]+2)+1));bands=[f'Dec{y}' for y in years]
  c,a=GRIDS[s,'JRC']
  ll=gpd.GeoSeries([GEOM[s]],crs=CRS).to_crs(4326).iloc[0]
  region=ee.Geometry.Rectangle(list(ll.bounds),proj='EPSG:4326',geodesic=False)
  jc=ee.ImageCollection(JID).filterBounds(region).sort('system:index')
  for i in range(jc.size().getInfo()):
   info=ee.Image(jc.toList(jc.size()).get(i)).select(bands).getInfo()['bands']
   for b in info:assert samegrid((c,a),(b['crs'],Affine(*b['crs_transform'])))
  img=jc.select(bands).mosaic().setDefaultProjection(c,list(a)[:6]).unmask(255).toUint8()
  for (row,col),z in g.assign(tr=(g.row//TILE)*TILE,tc=(g.col//TILE)*TILE).groupby(['tr','tc']):
   row=int(row);col=int(col);p=YDIR/f'{s}_{row}_{col}.parquet'
   if p.is_file() and not FORCE_REBUILD:q=pd.read_parquet(p)
   else:
    tr=a*Affine.translation(col,row)
    grid=dict(dimensions=dict(width=TILE,height=TILE),affineTransform=dict(scaleX=tr.a,shearX=tr.b,translateX=tr.c,scaleY=tr.e,shearY=tr.d,translateY=tr.f),crsCode=c)
    for attempt in range(5):
     try:arr=ee.data.computePixels(dict(expression=img,fileFormat='NUMPY_NDARRAY',grid=grid));break
     except Exception:
      if attempt==4:raise
      time.sleep(2**attempt)
    rr=z.row.to_numpy(int)-row;cc=z.col.to_numpy(int)-col;q=z[['site','row','col']].copy()
    for y,b in zip(years,bands):q[f'offset_{y-YEARS[s]}']=arr[b][rr,cc]
    atomic_parquet(p,q)
   assert q[['site','row','col']].reset_index(drop=True).equals(z[['site','row','col']].reset_index(drop=True))
   parts.append(q)
 Y=pd.concat(parts,ignore_index=True);atomic_parquet(YFILE,Y)
Y=J[['site','row','col','cover_pp','w','block500','history_valid','jrc_state','x','y']].merge(Y,on=['site','row','col'],validate='one_to_one')
assert len(Y)==len(J)
assert np.array_equal(Y.loc[Y.history_valid,'offset_-1'].to_numpy(),Y.loc[Y.history_valid,'jrc_state'].to_numpy())
# Within each site use identical complete-case spatial support across all available offsets.
YR=[]
for s,g in Y.groupby('site'):
 cols=[f'offset_{o}' for o in range(-2,min(2,2025-YEARS[s])+1)]
 q=g[g.history_valid & g[cols].isin(range(1,7)).all(axis=1)]
 for c in cols:
  off=int(c.split('_')[1])
  for state in [1,2,3,4,5,6]:
   z=q[q[c].eq(state)]
   YR.append(dict(site=s,offset=off,state=state,n=len(z),blocks=z.block500.nunique(),mean=z.cover_pp.mean(),median=z.cover_pp.median(),area_m2=z.w.sum()))
YR=pd.DataFrame(YR);YR.to_csv(FINAL/'jrc_year_sensitivity.csv',index=False)
print('07A complete — temporal sensitivity uses fixed common support within each site.')


In [ ]:
#@title 07B — Agreement, bounded patches and landscape relationships
# All computations below are offline from verified native tables.
RNG_SEED=2271;NBOOT=500;AGE_BINS=[0,3,6,9,12,15,20,26,40]
def wm(v,w):
 v=np.asarray(v,float);w=np.asarray(w,float);k=np.isfinite(v)&np.isfinite(w)&(w>0)
 return np.average(v[k],weights=w[k]) if k.any() else np.nan
def block_stats(g,value='cover_pp'):
 z=g[[value,'block500']].dropna();b=z.groupby('block500')[value].agg(['sum','count']).to_numpy(float)
 lo=hi=np.nan
 if len(b)>=5:
  rng=np.random.default_rng(RNG_SEED);u=b[rng.integers(0,len(b),(NBOOT,len(b)))].sum(axis=1);lo,hi=np.quantile(u[:,0]/u[:,1],[.025,.975])
 return dict(n=len(z),blocks=len(b),mean=z[value].mean(),median=z[value].median(),lo=lo,hi=hi)
def contrasts(d,category,low,high,label):
 rows=[]
 for s,g in d.groupby('site'):
  l=g[g[category].eq(low)];h=g[g[category].eq(high)]
  # Resample shared spatial blocks jointly; each category need >=5 occupied blocks.
  ids=sorted(set(l.block500)|set(h.block500));lo=hi=np.nan
  if l.block500.nunique()>=5 and h.block500.nunique()>=5:
   a=l.groupby('block500').cover_pp.agg(['sum','count']).reindex(ids,fill_value=0).to_numpy(float)
   b=h.groupby('block500').cover_pp.agg(['sum','count']).reindex(ids,fill_value=0).to_numpy(float)
   rng=np.random.default_rng(RNG_SEED);ix=rng.integers(0,len(ids),(NBOOT,len(ids)));aa=a[ix].sum(axis=1);bb=b[ix].sum(axis=1);ok=(aa[:,1]>0)&(bb[:,1]>0)
   if ok.sum()>=NBOOT*.9:lo,hi=np.quantile(bb[ok,0]/bb[ok,1]-aa[ok,0]/aa[ok,1],[.025,.975])
  rows.append(dict(site=s,contrast=label,n_low=len(l),n_high=len(h),blocks_low=l.block500.nunique(),blocks_high=h.block500.nunique(),mean_low=l.cover_pp.mean(),mean_high=h.cover_pp.mean(),difference=h.cover_pp.mean()-l.cover_pp.mean(),lo=lo,hi=hi))
 return rows
DIF=pd.DataFrame(contrasts(G,'loss_status','no detected loss','prior loss','Hansen loss')+contrasts(J,'jrc_state',1,2,'JRC degraded vs undisturbed')+contrasts(A,'agreement','neither dated','both within 1 yr','Both agree vs neither')+contrasts(J,'history_group','single entry','multiple entries','Multiple vs single entry'))
DIF.to_csv(FINAL/'primary_contrasts.csv',index=False)
# Timing differences compare only paired dated events; first/latest are distinct hypotheses.
timing=[]
for s,g in A.groupby('site'):
 for c in ['delta_first','delta_last']:
  z=g[c].dropna()
  timing.append(dict(site=s,comparison=c,n=len(z),median_signed=z.median(),median_absolute=z.abs().median(),within1=(z.abs()<=1).mean() if len(z) else np.nan))
TIME=pd.DataFrame(timing);TIME.to_csv(FINAL/'timing_agreement.csv',index=False)
# Bounded-span patches: deterministic oldest-year seed, grow adjacent eligible pixels whose
# dates lie within seed..seed+span. No transitive chaining beyond span; partition can depend on seed order.
# Observed fragments, not full landscape events; minimum 0.5 ha is sensitivity, not ecological truth.
PATCHFILE=FINAL/'bounded_patches.parquet'
if PATCHFILE.is_file() and not FORCE_REBUILD:
 BP=pd.read_parquet(PATCHFILE)
else:
 rows=[]
 for label,d,year in [('Hansen',G,'loss_year'),('JRC latest',J,'last_year'),('Agreement',A[A.agreement.eq('both within 1 yr')],'loss_year')]:
  for s,g in d[d[year].notna()].groupby('site'):
   g=g.sort_values([year,'row','col']).reset_index(drop=True)
   rr=g.row.to_numpy(int);cc=g.col.to_numpy(int);yy=g[year].to_numpy(int);w=g.w.to_numpy(float);cv=g.cover_pp.to_numpy(float);xx=g.x.to_numpy(float);xy=g.y.to_numpy(float)
   lookup={(r,c):i for i,(r,c) in enumerate(zip(rr,cc))}
   for span in [0,1,2,3]:
    seen=set();pid=0
    for i in range(len(g)):
     if i in seen:continue
     seen.add(i);queue=deque([i]);idx=[]
     while queue:
      j=queue.popleft();idx.append(j)
      for dr in [-1,0,1]:
       for dc in [-1,0,1]:
        k=lookup.get((rr[j]+dr,cc[j]+dc))
        if k is not None and k not in seen and yy[i]<=yy[k]<=yy[i]+span:
         seen.add(k);queue.append(k)
     k=np.asarray(idx);pid+=1;x=wm(xx[k],w[k]);y=wm(xy[k],w[k])
     rows.append(dict(site=s,variable=label,span=span,patch_id=pid,n=len(k),patch_ha=w[k].sum()/10000,cover_pp=wm(cv[k],w[k]),age=YEARS[s]-wm(yy[k],w[k]),year_min=yy[k].min(),year_max=yy[k].max(),x=x,y=y,block500=f'{int(x//500)}_{int(y//500)}'))
    assert len(seen)==len(g)
 BP=pd.DataFrame(rows);atomic_parquet(PATCHFILE,BP)
assert (BP.year_max-BP.year_min<=BP.span).all()
PS=BP.groupby(['site','variable','span']).patch_ha.agg(n='size',median_ha='median',max_ha='max').reset_index()
PS.to_csv(FINAL/'patch_span_sensitivity.csv',index=False)
# Main age comparison: same native matched support, all Hansen dated vs subset with agreement.
# Use 150 m cells with >=50% observed grid area, >=50% dated fraction, age IQR <=2.
AG=[]
for (s,gx,gy),g in A.assign(gx=(A.x//150).astype(int),gy=(A.y//150).astype(int)).groupby(['site','gx','gy']):
 if g.w.sum()<.5*150**2:continue
 for label,z in [('All Hansen dated',g[g.loss_year.notna()]),('Both agree',g[g.agreement.eq('both within 1 yr')])]:
  if not len(z) or z.w.sum()/g.w.sum()<.5 or z.age.quantile(.75)-z.age.quantile(.25)>2:continue
  AG.append(dict(site=s,cohort=label,age=z.age.median(),cover_pp=wm(z.cover_pp,z.w),block500=f'{int((gx+.5)*150//500)}_{int((gy+.5)*150//500)}',gx=gx,gy=gy))
AG=pd.DataFrame(AG,columns=['site','cohort','age','cover_pp','block500','gx','gy'])
def age_stats(d,group):
 rows=[];q=d.copy();q['age_bin']=pd.cut(q.age,AGE_BINS,right=False)
 for key,g in q.groupby(group+['age_bin'],observed=True):
  rows.append(dict(zip(group+['age_bin'],key),age=g.age.median(),**block_stats(g)))
 return pd.DataFrame(rows,columns=group+['age_bin','age','n','blocks','mean','median','lo','hi'])
AC=age_stats(AG,['site','cohort']);AC.to_csv(FINAL/'agreement_age_150m.csv',index=False)
AP=age_stats(BP[(BP.span==2)&(BP.patch_ha>=.5)&BP.variable.isin(['Hansen','Agreement'])],['site','variable']);AP.to_csv(FINAL/'agreement_patch_age.csv',index=False)
# All eligible cells, including zero cover and no observed disturbance. Product-specific native supports.
Q=T[T.eligible&T.scale.isin([90,150,300,500])&T.variable.isin(['Hansen_loss','JRC_degradation','JRC_last'])].copy()
LR=[]
for (s,var,scale),g in Q.groupby(['site','variable','scale']):
 for predictor in ['event_fraction','event_fraction_5yr','event_fraction_10yr','event_fraction_20yr']:
  z=g[[predictor,'cover_pp','block500']].dropna()
  rho=spearmanr(z[predictor],z.cover_pp).statistic if len(z)>2 and z[predictor].nunique()>1 and z.cover_pp.nunique()>1 else np.nan
  LR.append(dict(site=s,variable=var,scale=scale,predictor=predictor,n=len(z),blocks=z.block500.nunique(),rho=rho))
LR=pd.DataFrame(LR);LR.to_csv(FINAL/'landscape_scale_stats.csv',index=False)
# Agreement fraction on matched native support; avoid treating unmatched pixels as negative.
AF=[]
for (s,gx,gy),g in A.assign(gx=(A.x//150).astype(int),gy=(A.y//150).astype(int)).groupby(['site','gx','gy']):
 if g.w.sum()<.5*150**2:continue
 AF.append(dict(site=s,gx=gx,gy=gy,cover_pp=wm(g.cover_pp,g.w),agreement_fraction=wm(g.agreement.eq('both within 1 yr'),g.w),neither_fraction=wm(g.agreement.eq('neither dated'),g.w),x=wm(g.x,g.w),y=wm(g.y,g.w),block500=f'{int((gx+.5)*150//500)}_{int((gy+.5)*150//500)}'))
AF=pd.DataFrame(AF,columns=['site','gx','gy','cover_pp','agreement_fraction','neither_fraction','x','y','block500'])
AF.to_csv(FINAL/'agreement_landscape_150m.csv',index=False)
# Candidate high-cover, neither-dated locations; map observed discrepancy, not proven missed disturbance.
U=AF[(AF.neither_fraction>=.8)&(AF.cover_pp>=40)].copy()
U=U.sort_values('cover_pp',ascending=False).drop_duplicates(['site','block500'])
if len(U):
 ll=gpd.GeoSeries(gpd.points_from_xy(U.x,U.y),index=U.index,crs=CRS).to_crs(4326);U['longitude']=ll.x;U['latitude']=ll.y
U.to_csv(FINAL/'high_cover_neither_candidates.csv',index=False)
display(DIF.round(2));display(TIME.round(2));print('07B complete')


In [ ]:
#@title 07C — Six hypothesis-led figures and concise Word report
%pip -q install python-docx
from docx import Document
from docx.shared import Inches,Pt
from IPython.display import Markdown
figpaths=[]
def finish(fig,name):
 p=FINAL/f'{name}.png';fig.savefig(p,dpi=180,bbox_inches='tight');figpaths.append(p);plt.show();plt.close(fig)
# Figure 1: disturbance occurrence and product agreement (native boxplots; counts of spatial blocks).
fig,axs=plt.subplots(2,4,figsize=(15,7),sharey=True,layout='constrained')
for i,s in enumerate(SITES):
 for j,(d,col,labels) in enumerate([(G,'loss_status',[('no detected loss','No loss'),('prior loss','Loss')]),(J,'jrc_state',[(1,'Undist.'),(2,'Degraded')])]):
  ax=axs[j,i];g=d[d.site.eq(s)];vals=[];names=[]
  for key,label in labels:
   z=g[g[col].eq(key)]
   if len(z):vals.append(z.cover_pp.to_numpy());names.append(f'{label}\n{len(z):,} px / {z.block500.nunique()} b')
  if vals:ax.boxplot(vals,tick_labels=names,showfliers=False)
  ax.set_title(s if j==0 else 'JRC condition');ax.set_ylim(0,100);ax.tick_params(axis='x',labelsize=7)
  if i==0:ax.set_ylabel('Musanga cover (%)')
fig.suptitle('H1 — Musanga is enriched in mapped disturbed forest');finish(fig,'01_disturbance_boxplots')
# Figure 2: agreement categories and signed timing mismatch.
fig,axs=plt.subplots(2,4,figsize=(15,7),layout='constrained')
cats=['neither dated','Hansen only','JRC only','both different years','both within 1 yr'];short=['Neither','H only','J only','Both diff.','Both ±1']
for i,s in enumerate(SITES):
 g=A[A.site.eq(s)];ax=axs[0,i];vals=[];names=[]
 for c,l in zip(cats,short):
  z=g[g.agreement.eq(c)]
  if len(z):vals.append(z.cover_pp.to_numpy());names.append(f'{l}\n{len(z):,}')
 if vals:ax.boxplot(vals,tick_labels=names,showfliers=False)
 ax.set(title=s,ylim=(0,100));ax.tick_params(axis='x',rotation=30,labelsize=7)
 ax=axs[1,i]
 for c,l in [('delta_first','First JRC'),('delta_last','Latest JRC')]:
  z=g[c].dropna().to_numpy()
  if len(z):ax.hist(z,bins=np.arange(-36.5,37.5,1),density=True,histtype='step',label=l)
 ax.axvspan(-1.5,1.5,color='grey',alpha=.12);ax.set(xlabel='Hansen − JRC year',ylabel='Density')
 if ax.get_legend_handles_labels()[0]:ax.legend(fontsize=7)
axs[0,0].set_ylabel('Musanga cover (%)');fig.suptitle('H2 — Concordant disturbance products and timing uncertainty');finish(fig,'02_product_agreement')
# Figure 3: agreement-filtered age curves at 150 m and bounded patches.
fig,axs=plt.subplots(2,4,figsize=(15,7),sharey=True,sharex=True,layout='constrained')
for i,s in enumerate(SITES):
 for j,(tab,key,labels) in enumerate([(AC,'cohort',['All Hansen dated','Both agree']),(AP,'variable',['Hansen','Agreement'])]):
  ax=axs[j,i]
  for l in labels:
   z=tab[(tab.site==s)&(tab[key]==l)].sort_values('age')
   if z.empty:continue
   x=z.age.to_numpy(float);y=z['mean'].to_numpy(float);ok=z.blocks.to_numpy()>=5
   ax.plot(x,y,'-',alpha=.5,label=l);ax.scatter(x[ok],y[ok]);ax.scatter(x[~ok],y[~ok],facecolors='none',edgecolors='grey')
   ax.fill_between(x,z.lo.to_numpy(float),z.hi.to_numpy(float),alpha=.12)
  ax.set(title=s+(' — 150 m' if j==0 else ' — patches'),xlabel='Years since mapped event',ylim=(0,100))
  if ax.lines:ax.legend(fontsize=7)
  if i==0:ax.set_ylabel('Mean Musanga cover (%)')
fig.suptitle('H3 — Does dating agreement reveal an age trajectory? Hollow points: <5 blocks');finish(fig,'03_age_chronosequences')
# Figure 4: landscape fraction scatterplots, main scale 150 m.
fig,axs=plt.subplots(3,4,figsize=(15,10),sharex=True,sharey=True,layout='constrained')
for i,s in enumerate(SITES):
 for j,var in enumerate(['Hansen_loss','JRC_degradation','agreement']):
  ax=axs[j,i]
  if var=='agreement':z=AF[AF.site.eq(s)];x='agreement_fraction'
  else:z=Q[(Q.site==s)&(Q.variable==var)&(Q.scale==150)];x='event_fraction'
  ax.scatter(100*z[x],z.cover_pp,s=8,alpha=.35,rasterized=True)
  if len(z):
   zz=z.assign(bin=pd.cut(z[x],[-.001,.01,.25,.5,.75,1.001])).groupby('bin',observed=True).agg(x=(x,'mean'),y=('cover_pp','mean'))
   ax.plot(100*zz.x,zz.y,'o-',color='black',markersize=3)
  ax.set(title=f'{s} | n={len(z)}',xlim=(0,100),ylim=(0,100))
  if i==0:ax.set_ylabel(var.replace('_',' ')+'\nMusanga cover (%)')
  if j==2:ax.set_xlabel('Mapped fraction (%)')
fig.suptitle('H4 — Landscape disturbance fraction versus Musanga abundance (150 m)');finish(fig,'04_landscape_fraction')
# Figure 5: recurrence and temporal reference sensitivity.
fig,axs=plt.subplots(2,4,figsize=(15,7),sharey=True,layout='constrained')
for i,s in enumerate(SITES):
 ax=axs[0,i];g=J[J.site.eq(s)];vals=[];names=[]
 for c,l in [('no observed entry','None'),('single entry','Single'),('multiple entries','Multiple'),('left-censored','Censored')]:
  z=g[g.history_group.eq(c)]
  if len(z):vals.append(z.cover_pp.to_numpy());names.append(f'{l}\n{len(z):,}')
 if vals:ax.boxplot(vals,tick_labels=names,showfliers=False)
 ax.set(title=s,ylim=(0,100));ax.tick_params(axis='x',rotation=25,labelsize=7)
 ax=axs[1,i]
 for st,l in [(1,'Undisturbed'),(2,'Degraded')]:
  z=YR[(YR.site==s)&(YR.state==st)&(YR.n>0)].sort_values('offset')
  ax.plot(z.offset,z['mean'],'o-',label=l)
 ax.axvline(-1,ls='--',color='grey');ax.axvline(0,ls=':',color='grey');ax.set(xlabel='JRC year relative to UAV',ylim=(0,100))
 if ax.lines:ax.legend(fontsize=7)
axs[0,0].set_ylabel('Musanga cover (%)');axs[1,0].set_ylabel('Mean Musanga cover (%)')
fig.suptitle('H5 — Recurrence and sensitivity to JRC reference year');finish(fig,'05_recurrence_year')
# Figure 6: discrepancy maps in actual UTM coordinates, no background image or inference outside UAV.
fig,axs=plt.subplots(1,4,figsize=(15,5),layout='constrained')
for s,ax in zip(SITES,axs):
 z=AF[AF.site.eq(s)];u=U[U.site.eq(s)]
 ax.scatter(z.x,z.y,c='lightgrey',s=4);ax.scatter(u.x,u.y,c='darkred',s=18,label='Candidate block')
 ax.set(title=f'{s} | {len(u)} candidates',aspect='equal');ax.ticklabel_format(style='plain',useOffset=False);ax.tick_params(labelsize=6)
fig.suptitle('H6 — High Musanga with neither product dated (candidate locations, not validation)');finish(fig,'06_discrepancy_maps')
# Report: compact, hypothesis bullets with computed stats rather than predetermined outcomes.
doc=Document();style=doc.styles['Normal'];style.font.name='Calibri';style.font.size=Pt(10)
doc.add_heading('Musanga and forest disturbance legacies',0)
doc.add_paragraph('NB09 • V5 • Final ecological synthesis • Cross-sectional UAV chronosequence')
doc.add_heading('Data and methodology',1)
for t in [
 'Response: NB07 binary canopy maps (probability threshold 0.5), six mosaics across four forests. Native Hansen and JRC footprints sampled at 30 × 30 subpixel centres; at least 90% valid UAV/ROI support. Original ROI eroded by 100 m. The response is estimated canopy cover, not independently validated species abundance at every location.',
 f'Hansen: {GID}; JRC: {JID}. Forest condition uses the year before the configured UAV acquisition; events during the survey year are excluded. Configured years: '+', '.join(f'{s} {YEARS[s]}' for s in SITES)+'. Authoritatively verified sites: '+(', '.join(VERIFIED_FLIGHT_YEARS) or 'none; dates remain provisional')+'.',
 'Disturbance age is survey year minus mapped event year. JRC entries are forest-to-degraded or forest-to-deforested transitions; repeated entries are not independently confirmed events. No dated entry does not mean undisturbed. Left-censored histories are retained separately.',
 'Native products are paired by mutual nearest centres within 25 m (not exact overlap). Landscape units use centre assignment to fixed metric grids, area × valid support weights, >=50% observed grid area and >=90% valid history in product-specific grids. Primary 150 m; 90, 300 and 500 m sensitivity. Agreement grids require >=50% matched observed area.',
 'Age curves require >=50% dated fraction and age IQR <=2 years. Patches use 8-neighbour connectivity with bounded 0–3 year spans; primary sensitivity span 2 years and >=0.5 ha. Patches are UAV-clipped fragments, not independent events. 500 m block bootstrap intervals shown only with >=5 contributing blocks per group; results are descriptive and forest-specific.'
]:doc.add_paragraph(t)
def pic(i):
 doc.add_picture(str(figpaths[i]),width=Inches(6.4))
def lines_for(label):
 out=[]
 for _,z in DIF[DIF.contrast.eq(label)].iterrows():
  ci=f'; block CI [{z.lo:.1f}, {z.hi:.1f}]' if np.isfinite(z.lo) else '; insufficient block support for CI'
  out.append(f'{z.site}: {z.difference:+.1f} pp ({int(z.blocks_low)}/{int(z.blocks_high)} blocks{ci}).')
 return ' '.join(out)
doc.add_heading('Hypotheses and evidence',1)
doc.add_paragraph('H1 — Musanga is enriched in disturbed forest. '+lines_for('Hansen loss')+' JRC degraded versus undisturbed: '+lines_for('JRC degraded vs undisturbed'),style='List Bullet');pic(0)
doc.add_paragraph('H2 — Concordant products identify stronger disturbance legacies. '+lines_for('Both agree vs neither')+' Agreement is not independent validation; both products rely on Landsat histories.',style='List Bullet')
doc.add_paragraph('Timing agreement (within one year, conditional on both dated): '+'; '.join(f'{z.site}, {z.comparison.replace("delta_","")}: {100*z.within1:.0f}%' for _,z in TIME.iterrows() if np.isfinite(z.within1))+'.');pic(1)
doc.add_paragraph('H3 — A succession-age relationship may emerge after filtering temporal disagreement and aggregating patches. Inspect the age curves and their support; no monotonic or hump-shaped trajectory is assumed. Age bins with fewer than five blocks are exploratory. A curve is not an individual-tree trajectory.',style='List Bullet')
doc.add_paragraph('Patch fragmentation sensitivity (exact-year → two-year span, Hansen): '+'; '.join(f'{s}: {int(z.loc[z.span.eq(0),"n"].sum())} → {int(z.loc[z.span.eq(2),"n"].sum())}' for s,z in PS[PS.variable.eq('Hansen')].groupby('site'))+'.');pic(2)
doc.add_paragraph('H4 — Landscape disturbance extent may explain cover more robustly than a single age. At 150 m, Spearman rho for dated fraction: '+'; '.join(f'{z.site} {z.variable}: {z.rho:.2f} (n={z.n}, blocks={z.blocks})' for _,z in LR[(LR.scale==150)&(LR.predictor=='event_fraction')].iterrows() if np.isfinite(z.rho))+'. Correlations are descriptive; full scale/window sensitivities are in the CSV.',style='List Bullet');pic(3)
doc.add_paragraph('H5 — Multiple mapped entries may maintain or interrupt Musanga dominance. Multiple minus single: '+lines_for('Multiple vs single entry')+' Timing sensitivity compares fixed common support; post-survey classifications are diagnostics only.',style='List Bullet')
doc.add_paragraph('Undisturbed-class mean cover across available JRC offsets: '+'; '.join(f'{s}: '+', '.join(f'{int(z.offset):+d} yr {z["mean"]:.1f}%' for _,z in g.sort_values('offset').iterrows()) for s,g in YR[(YR.state==1)&(YR.n>0)].groupby('site'))+'.');pic(4)
doc.add_paragraph('H6 — High Musanga without dated disturbance may indicate undocumented legacies, dating errors or cover-estimation bias. Candidate blocks with >=80% neither dated and >=40% cover: '+', '.join(f'{s} {int((U.site==s).sum())}' for s in SITES)+'. These require independent historical imagery review; they are not confirmed omissions.',style='List Bullet');pic(5)
doc.add_heading('Conclusions and limits',1)
doc.add_paragraph('Prioritize the disturbance occurrence and concordance contrasts where spatial support is adequate. Treat age, recurrence and discrepancy hypotheses as conditional on acquisition-date verification, forest-specific history, UAV estimation error and spatial support. No causal or regional inference is warranted from four forests. No spectral analysis is included.')
report=FINAL/'NB09_final_ecological_summary.docx';doc.save(report)
atomic_json(FINAL/'completion.json',dict(complete=True,report=str(report),figures=[p.name for p in figpaths],flight_years_verified=VERIFIED_FLIGHT_YEARS,imagery_validation_completed=False))
display(Markdown('### Final NB09 summary\n'+ '\n'.join('- **'+l+'**: '+lines_for(l) for l in ['Hansen loss','JRC degraded vs undisturbed','Both agree vs neither','Multiple vs single entry'])))
print('NB09 FINAL REPORT:',report)
